In [0]:
%python
# Usamos la ruta que copiaste directamente del sistema
ruta_archivo = "/Volumes/main/default/datos_brutos/instagram_usage_lifestyle.csv"

# Cargamos los datos
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(ruta_archivo)

# --- INICIO DEL DATA PROFILING PROFUNDO (Fase 1) ---
print(f"✅ ¡Conexión establecida con el archivo en 'main'!")
print(f"📊 Total de registros en Bronze: {df.count()}")

# Mostramos el esquema para ver el ADN de tus datos
df.printSchema()

# Mostramos los primeros datos para validar que no haya basura
display(df.limit(10))

In [0]:
%python
from pyspark.sql.functions import col, count, when

# 1. Conteo de nulos corregido (Funciona para fechas, strings y números)
print("🔍 Buscando vacíos en el millón y medio de filas (Capa Bronze)...")

# Solo verificamos isNull() que es universal
nulos_df = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])

display(nulos_df)

# 2. Vamos a ver las estadísticas de las columnas numéricas clave
# Esto nos dirá si hay edades de 500 años o likes negativos (Outliers)
print("📊 Generando estadísticas descriptivas...")
display(df.select("age", "daily_active_minutes_instagram", "likes_given_per_day", "user_engagement_score").describe())

In [0]:
%python
# 1. ¿Cuántos países y géneros únicos tenemos? (Cardinalidad)
from pyspark.sql.functions import countDistinct

display(df.select(
    countDistinct("country").alias("Paises_Unicos"),
    countDistinct("gender").alias("Generos_Unicos"),
    countDistinct("income_level").alias("Niveles_Ingreso")
))

# 2. Vamos a ver el TOP 10 de países con más usuarios
# Esto nos dirá si los datos están balanceados
print("📍 Distribución por País (Top 10):")
df.groupBy("country").count().orderBy("count", ascending=False).show(10)

In [0]:
%python
# Usamos un comando SQL para crear el esquema 'silver' dentro de nuestro catálogo
spark.sql("CREATE SCHEMA IF NOT EXISTS adb_prof_socialmedia.silver")

# Verificamos que ahora aparezca en la lista
spark.sql("SHOW SCHEMAS IN adb_prof_socialmedia").show()

In [0]:
%python
from pyspark.sql.functions import col

# 1. Volvemos a definir la ruta y el DF (por si se borró la memoria)
ruta_archivo = "/Volumes/main/default/datos_brutos/instagram_usage_lifestyle.csv"
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(ruta_archivo)

# 2. Ahora sí, aplicamos la lógica de limpieza para Silver
# Filtramos nulos en país para asegurar calidad
df_silver = df.filter(col("country").isNotNull())

# 3. Guardamos como TABLA oficial
print("🚀 Escribiendo 1.5M de registros en la Capa Silver...")

df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("country") \
    .saveAsTable("adb_prof_socialmedia.silver.instagram_lifestyle")

print("✅ ¡Misión cumplida! Tabla 'instagram_lifestyle' creada en la Capa Silver.")

In [0]:
%sql
-- Vamos a ver cuántos usuarios tenemos por nivel de ingresos en Colombia (o el país que elijas)
-- Gracias al particionamiento, esto debería ser veloz como un rayo
SELECT 
    income_level, 
    count(*) as total_usuarios,
    round(avg(daily_active_minutes_instagram), 2) as promedio_minutos_dia
FROM adb_prof_socialmedia.silver.instagram_lifestyle
WHERE country = 'United States' -- Prueba con uno de los países que vimos en el top
GROUP BY income_level
ORDER BY total_usuarios DESC;

In [0]:
%python
# Creamos el esquema GOLD
spark.sql("CREATE SCHEMA IF NOT EXISTS adb_prof_socialmedia.gold")

# Listamos para confirmar que el "Trío de Oro" (Bronze, Silver, Gold) está completo
spark.sql("SHOW SCHEMAS IN adb_prof_socialmedia").show()

In [0]:
%python
from pyspark.sql.functions import avg, count, round, col

# 1. Leemos de nuestra capa Silver (la que ya está limpia y particionada)
df_silver = spark.table("adb_prof_socialmedia.silver.instagram_lifestyle")

# 2. Creamos el modelo "Gold": Agrupamos por País y Nivel de Ingresos
# Calculamos promedios de estrés, sueño y tiempo en Instagram
df_gold = df_silver.groupBy("country", "income_level").agg(
    count("user_id").alias("total_usuarios"),
    round(avg("perceived_stress_score"), 2).alias("estres_promedio"),
    round(avg("sleep_hours_per_night"), 2).alias("horas_sueno_promedio"),
    round(avg("daily_active_minutes_instagram"), 2).alias("minutos_ig_promedio")
).orderBy("country", "income_level")

# 3. Guardamos en la Capa Gold como una tabla permanente
df_gold.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("adb_prof_socialmedia.gold.reporte_salud_engagement")

print("🔥 ¡Tabla GOLD creada con éxito! Lista para Power BI o Dashboards.")

# Mostramos el resultado final
display(spark.table("adb_prof_socialmedia.gold.reporte_salud_engagement"))

In [0]:
%sql
-- 1. Añadimos un comentario a la tabla para el Catálogo
COMMENT ON TABLE adb_prof_socialmedia.gold.reporte_salud_engagement 
IS 'Tabla final de KPIs que cruza salud mental con uso de Instagram. Uso exclusivo para Marketing y RRHH.';

-- 2. Documentamos las columnas clave (Esto aparecerá en la interfaz de Azure)
ALTER TABLE adb_prof_socialmedia.gold.reporte_salud_engagement 
ALTER COLUMN estres_promedio COMMENT 'Puntaje de estrés percibido (escala 0-40).';

ALTER TABLE adb_prof_socialmedia.gold.reporte_salud_engagement 
ALTER COLUMN minutos_ig_promedio COMMENT 'Tiempo promedio diario en la plataforma en minutos.';

In [0]:
%python
# Aplicamos una etiqueta de seguridad mediante Python (Simulando gobernanza)
# Nota: En entornos empresariales, esto permite filtrar quién ve la columna.
spark.sql("""
  ALTER TABLE adb_prof_socialmedia.gold.reporte_salud_engagement 
  SET TAGS ('sensibilidad' = 'alta', 'departamento' = 'finanzas')
""")

print("🔒 Capa de Gobernanza aplicada: Metadatos y Tags configurados.")

In [0]:
%sql
DESCRIBE TABLE EXTENDED adb_prof_socialmedia.gold.reporte_salud_engagement;